In [3]:
# 필요한 라이브러리들을 임포트합니다.
from sklearn.linear_model import LogisticRegression # 로지스틱 회귀 모델
from sklearn.ensemble import RandomForestClassifier # 랜덤 포레스트 분류기
from sklearn.model_selection import train_test_split, GridSearchCV # 훈련/테스트 데이터 분할 및 그리드 서치를 위한 모듈
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif # 특성 선택을 위한 모듈 (최고 K개 선택, 분산 임계값, ANOVA F-값)
from sklearn.tree import DecisionTreeClassifier # 결정 트리 분류기
from sklearn.metrics import roc_auc_score, fbeta_score, make_scorer # ROC AUC 점수, F-베타 점수, 커스텀 스코어러 생성
from xgboost import XGBClassifier # XGBoost 분류기
import shap # SHAP(SHapley Additive exPlanations) 라이브러리 (모델 예측 설명)
import matplotlib.pyplot as plt # 데이터 시각화를 위한 라이브러리

import pandas as pd # 데이터 조작 및 분석을 위한 라이브러리
import numpy as np # 수치 계산을 위한 라이브러리
import datetime as dt # 날짜 및 시간 처리를 위한 라이브러리
import json # JSON 데이터 처리를 위한 라이브러리

In [4]:
# 경고 메시지 처리를 위한 모듈
import warnings 

# 'use_label_encoder' 경고만 무시합니다.
warnings.filterwarnings("ignore")

#### prepare "data/initial_dataset.p"

In [5]:
# .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3

# # 파일 경로 지정
# file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.xlsx'

# # 엑셀 파일을 DataFrame으로 읽어오기
# # 기본적으로 첫 번째 시트를 읽어옵니다.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)

In [6]:
# .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress

# # 파일 경로 지정
# file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.xlsx'

# # 엑셀 파일을 DataFrame으로 읽어오기
# # 기본적으로 첫 번째 시트를 읽어옵니다.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)

In [7]:
# read *.p
pickle_file_path_1 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'
data_row_1 = pd.read_pickle(pickle_file_path_1)
pickle_file_path_2 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'
data_row_2 = pd.read_pickle(pickle_file_path_2)

In [8]:
# 필요한 컬럼만 keep

# 파일 경로 지정
file_path = 'data/cols_to_keep.csv'

# CSV 파일을 DataFrame으로 읽어오기
cols_to_keep_df = pd.read_csv(file_path)

cols_to_keep = cols_to_keep_df.iloc[:, 0].tolist()

data_row_1 = data_row_1[cols_to_keep]

In [9]:
# data_row <= data_row1 data_row2

# data_row_2에서 조인할 컬럼만 선택
columns_to_join = ['DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP',
                  #  'wafer_id', 
                #    'SensorOffsetHot-RoomAfterBake', 
                #    'SensorOffsetHot-ColdAfterBake', 
                   'BG pass/fail']

# 선택한 컬럼으로 data_row_2의 부분집합 DataFrame 생성
data_row_2_subset = data_row_2[columns_to_join]

# data_row_1에 data_row_2의 선택된 컬럼들을 조인 키 'DevID'로 병합
data_row = pd.merge(data_row_1, data_row_2_subset, on='DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP', how='left')

# # 결과 DataFrame 확인
# print(merged_df.head())

In [10]:
initial_dataset = data_row.copy() # 원본 데이터셋 복사
# processed_dataset = initial_dataset.copy() # 원본 데이터셋 복사

In [11]:
# prepare for target

initial_dataset['Pass/Fail_pass'] = ((initial_dataset['soft_bin of FT1'] == 1) &
                   (initial_dataset['soft_bin of FT2'] == 1) &
                   (initial_dataset['soft_bin'] == 1)).astype(int)

In [12]:
# prepare for base model

initial_dataset['band gap dpat'] = initial_dataset['BG pass/fail'].apply(lambda x: 'bandGapFail' if x == 'impossible wafer' else 'ok for band gap')

# 컬럼 이름 변경 딕셔너리 생성
new_column_names = {
    'wafer_id': 'WAFER_NO',
    'DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP': 'DevID'
}

# .rename() 메서드를 사용하여 컬럼 이름 변경 (inplace=True로 원본 데이터프레임에 바로 적용)
initial_dataset.rename(columns=new_column_names, inplace=True)

# 변경된 컬럼 이름 확인
# print(initial_dataset.columns)

In [13]:
# initial_dataset

In [14]:
# 제거할 컬럼 리스트 정의
columns_to_drop = [
    'soft_bin of FT1',
    'soft_bin of FT2',
    'soft_bin',
    'BG pass/fail'
]

# 컬럼 drop (원본 DataFrame을 변경하려면 inplace=True 사용)
# 또는 새로운 DataFrame을 만들려면 processed_dataset = processed_dataset.drop(...) 사용
initial_dataset.drop(columns=columns_to_drop, inplace=True)

In [15]:
# Rename the columns
initial_dataset.rename(columns={'x_pos': 'X', 'y_pos': 'Y'}, inplace=True)

In [16]:
# Radius 컬럼 계산
# np.sqrt() 함수는 각 요소의 제곱근을 계산합니다.
initial_dataset['Radius'] = np.sqrt(initial_dataset['X']**2 + initial_dataset['Y']**2)

In [17]:
initial_dataset.to_pickle("data/initial_dataset.p")

#### scnarios

In [18]:
# import vars and function

from algos.algos import *
from config.config import *

In [19]:
import copy

##### preprocess_dataset

In [20]:
# def preprocess_dataset(initial_dataset: pd.DataFrame):
    # return processed_dataset # 전처리된 데이터셋 반환

preprocessed_dataset = preprocess_dataset(initial_dataset)

##### create_train_and_test_data
# def create_train_test_data(
#     preprocessed_dataset: pd.DataFrame,
#     split_parameter: dict = None
# ):
#     return train_data, test_data, split_parameter_info



     데이터셋 전처리 중...
     전처리 완료!



##### create_train_and_test_data

In [21]:
# def create_train_test_data(
#     preprocessed_dataset: pd.DataFrame,
#     split_parameter: dict = None
# ):
#     return train_data, test_data, split_parameter_info

In [22]:
split_parameter_default

{'test_size': 0.2,
 'random_state': 42,
 'apply_filter_split': False,
 'var_threshold_split': 0.0,
 'corr_threshold_split': 0.98,
 'sampling_method': None,
 'sampling_ratio': None,
 'apply_feature_generation': False,
 'sum_features': False,
 'diff_features': False,
 'poly_features': False,
 'poly_degree': 2,
 'apply_filter_gen': False,
 'var_threshold_gen': 0.0,
 'corr_threshold_gen': 0.1}

In [23]:
split_parameter = copy.deepcopy(split_parameter_default)

In [24]:
train_data, test_data, split_parameter_info = create_train_test_data(preprocessed_dataset, split_parameter)



##############################################################################################################################
# 3) Create Train/Test Split (훈련/테스트 데이터 분할) 
##############################################################################################################################

     훈련 및 테스트 데이터셋 생성 중...
     - 분할 전 필터링 미적용.
     - Feature Generation 미적용.

     - 분할 전 훈련 데이터 클래스 분포: {0.0: 3546, 1.0: 71}
     - 샘플링 미적용


In [25]:
feature_selector_params_var = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_var["filter_methods"]["apply_variance_filter"] = True
feature_selector_params_var["filter_methods"]["var_threshold"] = 0.00
feature_selection_info_var = select_feature(train_data, feature_selector_params_var)


--- 피처 선택기: FeatureFilter ---

--- 피처 필터링 시작 ---
    - 분산 필터링 후 남은 피처 수: 1401

피처 선택 결과가 'data/result/jsons\feature_selection_info_250902_110618_76c51881.json' 파일에 저장되었습니다.

- 최종 피처 수: 1401


In [26]:
features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

value_type = "variance"

features_values = feature_selection_info_var["selection_details"][value_type]["features_values_checked"]

features_values_df = pd.DataFrame(
    list(features_values.items()), 
    columns=['feature_name', feature_selection_info_var["feature_selector_name"]+"_"+value_type]
)

features_values_dfs = pd.merge(
    features_values_dfs,
    features_values_df,
    how='outer',
    left_on='feature_name',
    right_on='feature_name'
    )

In [27]:
# target은 가장 오른쪽 열을 선택
target_df = train_data.iloc[:, -1]
# feature는 나머지 모든 열
features_df = train_data.iloc[:, :-1]

X_train, X_test, y_train, y_test = train_test_split(features_df, target_df, test_size=0.2, random_state=42)

In [28]:
feature_importance_df = features_values_dfs.copy()
feature_importance_df

,feature_name,FeatureFilter_variance
0,AC_COIL_FACTOR of 1103959_69_1133529_YPP,2.951574e-33
1,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,2.951574e-33
2,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5...,2.951574e-33
3,AC_COIL_FACTOR of 1103959_69_1133592_RPP,2.951574e-33
4,AC_GAIN_32 of 1103959_69_1133529_cp1,0.000000e+00
...,...,...
1646,Zb_V_OBVOL_VSS_L of 1103959_69_1133529_cp1_cp1...,1.000277e+00
1647,Zb_V_OBVOL_VSS_L of 1103959_69_1133529_cp1p5,1.000277e+00
1648,Zb_V_OBVOL_VSS_L of 1103959_69_1133592_QPP,1.000277e+00
1649,Zb_V_OBVOL_VSS_L of 1103959_69_JPP,1.000277e+00


In [29]:
### 적용

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import fbeta_score, make_scorer, confusion_matrix
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import uuid

In [30]:
# run_optimization_for_feature_importance : 특정 중요도 컬럼을 기준으로 피처를 선택하고 최적의 모델을 찾는 함수

def run_optimization_for_feature_importance(train_data, target_data, feature_importance_df, importance_column, k_percentiles):
    """
    특정 중요도 컬럼을 기준으로 피처를 선택하고 최적의 모델을 찾는 함수
    
    Parameters:
    - train_data (pd.DataFrame): 훈련 데이터
    - target_data (pd.Series): 타겟 데이터
    - feature_importance_df (pd.DataFrame): 피처 중요도 정보가 담긴 DataFrame
    - importance_column (str): 중요도 순위를 결정할 컬럼명
    - k_percentiles (list): 선택할 피처의 백분위수 후보 리스트 (예: [0.05, 0.1, 0.25, 0.5])

    Returns:
    - pd.DataFrame: 최적화된 피처 중요도 정보
    - pd.DataFrame: 모델 성능 요약 정보
    """
    f2_scorer = make_scorer(fbeta_score, beta=2.0)
    
    # 중요도 컬럼의 값에 따라 상위 K개의 피처를 선택
    sorted_features = feature_importance_df.sort_values(
        by=importance_column, ascending=False
    )['feature_name']
    
    # 백분위수를 실제 피처 개수로 변환
    n_features_total = len(sorted_features)
    k_values = [max(1, int(n_features_total * p)) for p in k_percentiles]
    
    best_k = k_values[0]
    best_score = -1.0
    best_pipeline = None
    selected_feature_list = []
    # ⭐️ 최적의 백분위수 값을 저장할 변수
    best_k_percentile = k_percentiles[0]

    for i, k in enumerate(k_values):
        top_k_features = sorted_features.head(k).tolist()
        
        # 최적 피처로만 구성된 데이터셋 준비
        X_train_filtered = train_data[top_k_features]
        
        # 모델 훈련 파이프라인 (SelectKBest 대신 피처 직접 선택)
        pipeline = Pipeline([
            ('model', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
        ])
        
        param_grid = {
            'model__n_estimators': [50, 100],
            'model__max_depth': [3, 5]
        }
        
        grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring=f2_scorer, n_jobs=-1)
        grid_search.fit(X_train_filtered, target_data)

        # 현재 K의 성능 평가
        if grid_search.best_score_ > best_score:
            best_score = grid_search.best_score_
            best_k = k
            best_pipeline = grid_search.best_estimator_
            selected_feature_list = top_k_features
            # ⭐️ 최적의 백분위수 업데이트
            best_k_percentile = k_percentiles[i]

    # 최적 모델의 피처 중요도 및 성능 정보 생성
    best_xgb_model = best_pipeline.named_steps['model']
    
    feature_info = pd.DataFrame({
        'feature_name': train_data.columns
    })
    feature_info['is_selected'] = feature_info['feature_name'].isin(selected_feature_list)
    feature_info['importance_column'] = importance_column
    feature_importances = {name: 0 for name in train_data.columns}
    
    # 선택된 피처에 대해서만 중요도 점수를 부여
    for i, importance in enumerate(best_xgb_model.feature_importances_):
        if i < len(selected_feature_list):
            feature_importances[selected_feature_list[i]] = importance
        
    feature_info['importance_score'] = feature_info['feature_name'].map(feature_importances)
    feature_info['feature_value_by_importance_column'] = feature_info['feature_name'].map(
        feature_importance_df.set_index('feature_name')[importance_column]
    )

    # 테스트 데이터로 최종 성능 평가
    y_pred = best_pipeline.predict(X_test[selected_feature_list])
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    performance_summary = pd.DataFrame([{
        'fn': fn,
        'fp': fp,
        'tn': tn,
        'tp': tp,
        'feature_selector_name': importance_column,
        'initial_feature_count': X_train.shape[1],
        'final_feature_count': len(selected_feature_list),
        'f2_score': fbeta_score(y_test, y_pred, beta=2.0),
        'importance_column': importance_column,
        # ⭐️ 최적 백분위수 컬럼 추가
        'best_k_percentile': best_k_percentile
    }])

    return feature_info, performance_summary

In [31]:
# importance_cols

In [32]:
# 중요도 컬럼별로 최적화 반복 수행 및 결과 누적
# 피처이름 컬럼 제외 2번째 컬럼 이후의 중요도 컬럼명을 리스트로 저장
importance_cols = feature_importance_df.columns[1:].tolist()

# ⭐️ 백분위수 후보 리스트로 변경 : 사용자가 정의함
# k_percentiles = [0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.85, 0.9]
k_percentiles = [0.75] # ⭐️ 최적의 변수선택 방법 적용 : 분산 기준 75퍼센타일의 변수선택

all_feature_infos = []
all_performance_summaries = []

for col in importance_cols:
    print(f"\n--- {col} 컬럼 기준 최적화 수행 ---")
    feat_info, perf_summary = run_optimization_for_feature_importance(
        X_train, y_train, feature_importance_df, col, k_percentiles
    )
    all_feature_infos.append(feat_info)
    all_performance_summaries.append(perf_summary)

final_feature_info_df = pd.concat(all_feature_infos, ignore_index=True)
final_feature_performance_summary_df = pd.concat(all_performance_summaries, ignore_index=True)

# 4. 결과 출력 및 CSV 저장
print("\n--- 최종 누적된 피처 중요도 정보 ---")
print(final_feature_info_df.head(10))
final_feature_info_df.to_csv('final_feature_info.csv', index=False)

print("\n--- 최종 누적된 모델 성능 요약 ---")
print(final_feature_performance_summary_df)
final_feature_performance_summary_df.to_csv('final_feature_performance_summary.csv', index=False)


--- FeatureFilter_variance 컬럼 기준 최적화 수행 ---

--- 최종 누적된 피처 중요도 정보 ---
                                        feature_name  is_selected  \
0                                                  X         True   
1                                                  Y         True   
2                                             Radius         True   
3           AC_COIL_FACTOR of 1103959_69_1133529_YPP        False   
4     AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5        False   
5  AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5...        False   
6           AC_COIL_FACTOR of 1103959_69_1133592_RPP        False   
7               AC_GAIN_32 of 1103959_69_1133529_cp1        False   
8     AC_GAIN_32 of 1103959_69_1133529_cp1_cp1p5_YPP        False   
9  AC_GAIN_32 of 1103959_69_1133529_cp1_cp1p5_YPP...        False   

        importance_column  importance_score  \
0  FeatureFilter_variance          0.000000   
1  FeatureFilter_variance          0.003406   
2  FeatureFilter_variance   

In [33]:
# 성능체크파이프라인 입력 준비 : feature_selection_results

feature_selection_results = {}

# Iterate line by line
for idx, row in final_feature_performance_summary_df.iterrows():
    feature_selection_result = {}
    feature_selection_result["feature_selector_idx"] = idx
    feature_selection_result["feature_selector_name"] = row['feature_selector_name']
    feature_selection_result["initial_feature_count"] = row['initial_feature_count']
    feature_selection_result["final_feature_count"] = row['final_feature_count']
    feature_selection_result.setdefault("Params", {})["f2_score"] = row['f2_score']
    feature_selection_result.setdefault("Params", {})["best_k_percentile"] = row['best_k_percentile']
    
    feature_name_list = final_feature_info_df[
        (final_feature_info_df["importance_column"] == row['feature_selector_name']) &
        (final_feature_info_df["is_selected"] == True)
    ]["feature_name"].tolist()
    feature_selection_result["final_features"] = feature_name_list

    feature_selection_results[idx] = feature_selection_result

In [34]:
# feature_selection_result

In [35]:
# feature_selection_result["final_features"]
feature_selection_info_var_final = feature_selection_result

##### 샘플링 방법 > 성능체크 : 여러 샘플링, var 변수선택, rf_cv 모델 파이프라인 활용

In [36]:
split_parameter_default

{'test_size': 0.2,
 'random_state': 42,
 'apply_filter_split': False,
 'var_threshold_split': 0.0,
 'corr_threshold_split': 0.98,
 'sampling_method': None,
 'sampling_ratio': None,
 'apply_feature_generation': False,
 'sum_features': False,
 'diff_features': False,
 'poly_features': False,
 'poly_degree': 2,
 'apply_filter_gen': False,
 'var_threshold_gen': 0.0,
 'corr_threshold_gen': 0.1}

###### change random seed, No oversampling

In [89]:
# split and sample test : pipeline
## 선택한 피처셋 feature_selection_info['final_features'] 리스트를 모델링 파이프라인에 적용, 모델링 결과를 리턴

def pl_get_model_result(
        preprocessed_dataset,
        split_parameter,
        feature_selection_info,
        train_parameters,
        ):
    
    train_data, test_data, split_parameter_info = create_train_test_data(preprocessed_dataset, split_parameter)
    trained_model, feature_importance, train_parameters_info = train_model_rf_cv(train_data.copy(), feature_selection_info, train_parameters)
    forecast_dataset, shap_values_random_forest = forecast(test_data, trained_model, feature_selection_info)
    train_dataset_proba, best_threshold = find_best_threshold(trained_model, train_data.copy(), feature_selection_info)
    roc_data, auc_score = roc_from_scratch(forecast_dataset, test_data, partitions=100)
    train_dataset_metrics = create_metrics_on_train(train_dataset_proba, best_threshold)
    metrics = create_metrics(forecast_dataset, test_data, auc_score, best_threshold)
    results = create_results(forecast_dataset, test_data, best_threshold)
    
    return \
        split_parameter_info, \
        trained_model, \
        feature_importance, \
        forecast_dataset, \
        train_dataset_proba, \
        best_threshold, \
        roc_data, \
        auc_score, \
        train_dataset_metrics, \
        metrics, \
        results

In [101]:
# create_train performance test - submit and summary result


def run_summary_pl_result_create_train_and_test(split_parameters, save_path):

    split_parameter_info_set = pd.DataFrame()
    trained_model_set = {}
    feature_importance_set = pd.DataFrame()
    forecast_dataset_set = pd.DataFrame()
    train_dataset_proba_set = pd.DataFrame()
    best_threshold_set = pd.DataFrame()
    roc_data_set = pd.DataFrame()
    auc_score_set = pd.DataFrame()
    train_dataset_metrics_set = pd.DataFrame()
    metrics_set = pd.DataFrame()
    results_set = pd.DataFrame()


    usr_pl_smpl_test_result_ftpn_df = pd.DataFrame()

    for split_parameter_idx, split_parameter in split_parameters.items():
        split_parameter_info, \
        trained_model, \
        feature_importance, \
        forecast_dataset, \
        train_dataset_proba, \
        best_threshold, \
        roc_data, \
        auc_score, \
        train_dataset_metrics, \
        metrics, \
        results \
        = \
        pl_get_model_result(
                preprocessed_dataset,
                split_parameter,
                feature_selection_info_var_final,
                train_parameters_list_default["rf_cv"],
                )

        # split_parameter_info_set["split_parameter_idx"] = split_parameter_idx
        # split_parameter_info_set = pd.concat([split_parameter_info_set, split_parameter_info], ignore_index=True)
        # split_parameter_info를 데이터프레임으로 변환하여 추가하는 로직
        split_parameter_info_df = pd.DataFrame([split_parameter_info])
        split_parameter_info_df["split_parameter_idx"] = split_parameter_idx
        split_parameter_info_set = pd.concat([split_parameter_info_set, split_parameter_info_df], ignore_index=True)    

        trained_model_set[split_parameter_idx] = trained_model

        feature_importance["split_parameter_idx"] = split_parameter_idx
        feature_importance_set = pd.concat([feature_importance_set, feature_importance], ignore_index=True)

        forecast_dataset_df = pd.DataFrame({
            "split_parameter_idx": [split_parameter_idx] * len(forecast_dataset),
            "forecast_prob": forecast_dataset
        })
        forecast_dataset_set = pd.concat([forecast_dataset_set, forecast_dataset_df], ignore_index=True)
        
        train_dataset_proba["split_parameter_idx"] = split_parameter_idx
        train_dataset_proba_set = pd.concat([train_dataset_proba_set, train_dataset_proba], ignore_index=True)

        train_dataset_proba["split_parameter_idx"] = split_parameter_idx
        train_dataset_proba_set = pd.concat([train_dataset_proba_set, train_dataset_proba], ignore_index=True)

        best_threshold_df = pd.DataFrame({
        "split_parameter_idx": [split_parameter_idx],
        "best_threshold": [best_threshold]
        })
        best_threshold_set = pd.concat([best_threshold_set, best_threshold_df], ignore_index=True)

        roc_data["split_parameter_idx"] = split_parameter_idx
        roc_data_set = pd.concat([roc_data_set, roc_data], ignore_index=True)

        auc_score_df = pd.DataFrame({
        "split_parameter_idx": [split_parameter_idx],
        "auc_score": [auc_score]
        })
        auc_score_set = pd.concat([auc_score_set, auc_score_df], ignore_index=True)

        train_dataset_metrics["split_parameter_idx"] = split_parameter_idx
        train_dataset_metrics_set = pd.concat([train_dataset_metrics_set, train_dataset_metrics], ignore_index=True)

        metrics_dict = {
            "f1_score": metrics["f1_score"],
            "recall": metrics["recall"],
            "precision": metrics["precision"],
            "accuracy": metrics["accuracy"],
            "auc_score": metrics["auc_score"],
            "tp": metrics["dict_ftpn"]["tp"],
            "tn": metrics["dict_ftpn"]["tn"],
            "fp": metrics["dict_ftpn"]["fp"],
            "fn": metrics["dict_ftpn"]["fn"],
            "number_of_predictions": metrics["number_of_predictions"],
            "number_of_good_predictions": metrics["number_of_good_predictions"],
            "number_of_false_predictions": metrics["number_of_false_predictions"],
            "split_parameter_idx": split_parameter_idx  # 현재 필터 이름 추가
        }
        metrics_df = pd.DataFrame([metrics_dict])
        metrics_set = pd.concat([metrics_set, metrics_df], ignore_index=True)

        results["split_parameter_idx"] = split_parameter_idx
        results_set = pd.concat([results_set, results], ignore_index=True)

        dict_ftpn = metrics["dict_ftpn"]
        if "class_distribution_after_sampling" in split_parameter_info :
            class_distribution = split_parameter_info["class_distribution_after_sampling"]
        else :
            class_distribution = split_parameter_info["class_distribution_before_sampling"]
        new_row = {
            'fn': dict_ftpn.get('fn'),
            'fp': dict_ftpn.get('fp'),
            'tn': dict_ftpn.get('tn'),
            'tp': dict_ftpn.get('tp'),
            'random_state' : split_parameter_info["random_state"],
            'sampling_method_used' : split_parameter_info["sampling_method_used"],
            'sampling_ratio_used' : split_parameter_info["sampling_ratio_used"],
            'class_distribution_before_sampling' : split_parameter_info["class_distribution_before_sampling"],
            'class_distribution_after_sampling' : class_distribution
        }
        
        usr_pl_smpl_test_result_ftpn_df = pd.concat([usr_pl_smpl_test_result_ftpn_df, pd.DataFrame([new_row])], ignore_index=True)

    import pickle
    import os

    # save_path = "data/result/create_train_test/split_random_seed_change"
    os.makedirs(save_path, exist_ok=True)

    dict_vars = [
        (trained_model_set, "trained_model_set")
    ]

    df_vars = [
        (split_parameter_info_set, "split_parameter_info_set"),
        (feature_importance_set, "feature_importance_set"),
        (forecast_dataset_set, "forecast_dataset_set"),
        (train_dataset_proba_set, "train_dataset_proba_set"),
        (best_threshold_set, "best_threshold_set"),
        (roc_data_set, "roc_data_set"),
        (auc_score_set, "auc_score_set"),
        (train_dataset_metrics_set, "train_dataset_metrics_set"),
        (metrics_set, "metrics_set"),
        (results_set, "results_set"),
        (usr_pl_smpl_test_result_ftpn_df, "usr_pl_smpl_test_result_ftpn_df")
    ]

    for var, name in dict_vars:
        file_path = os.path.join(save_path, f"{name}.pickle")
        with open(file_path, "wb") as f:
            pickle.dump(var, f)
        print(f"Saved {name} to {file_path}")

    for var, name in df_vars:
        file_path = os.path.join(save_path, f"{name}.csv")
        var.to_csv(file_path, index=False)
        print(f"Saved {name} to {file_path}")
    
    return usr_pl_smpl_test_result_ftpn_df

###### model performance check by random seed change

In [102]:
params = [
    [41, None, None],
    [42, None, None],
    [43, None, None],
    [44, None, None],
    [45, None, None],
    [46, None, None],
    [47, None, None],
    [48, None, None],
    [49, None, None],
    [50, None, None],

    # [49, "ROS", 1.0],
    # [49, "ROS", 2.0],
    # [49, "SMOTE", 1.0],
    # [49, "SMOTE", 2.0],
    # [49, "ADASYN", 1.0],
    # [49, "ADASYN", 2.0]
]

In [ ]:
split_parameters = {}
for i, param in enumerate(params):
    split_parameter = copy.deepcopy(split_parameter_default)
    split_parameter["random_state"] = param[0]
    split_parameter["sampling_method"] = param[1]
    split_parameter["sampling_ratio"] = param[2]
    split_parameters[i] = split_parameter

# print(split_parameters)

In [104]:
# for key, value in split_parameters.items():
#     print(f"Key: {key}")
#     print(f"Value: {value}")
#     print("-" * 20)

In [105]:
# split_parameters

In [106]:
# split_parameters.items()

In [107]:
# feature_selection_info_var_final

In [108]:
save_path = "data/result/create_train_test/split_random_seed_change"
usr_pl_smpl_test_result_ftpn_df_random_seed = run_summary_pl_result_create_train_and_test(split_parameters, save_path)



##############################################################################################################################
# 3) Create Train/Test Split (훈련/테스트 데이터 분할) 
##############################################################################################################################

     훈련 및 테스트 데이터셋 생성 중...
     - 분할 전 필터링 미적용.
     - Feature Generation 미적용.

     - 분할 전 훈련 데이터 클래스 분포: {0.0: 3546, 1.0: 71}
     - 샘플링 미적용
      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
    Best F2 (class=1) score (CV): 0.2451

      Forecasting the test dataset...
      Forecasting done!
Best threshold for F2 score: 0.7475 with F2 score: 0.6026
      Calculation of the ROC curve...
      Calculation done
      Scoring...
      Scoring done

      Creating the metrics...


######################

In [109]:
# 분할 랜덤시드 모델성능요약

usr_pl_smpl_test_result_ftpn_df_random_seed


,fn,fp,tn,tp,random_state,sampling_method_used,sampling_ratio_used,class_distribution_before_sampling,class_distribution_after_sampling
0,9,157,730,9,41,None,None,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 71}"
1,6,186,701,12,42,None,None,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 71}"
2,4,246,641,14,43,None,None,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 71}"
3,4,175,712,14,44,None,None,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 71}"
4,6,168,719,12,45,None,None,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 71}"
5,4,132,755,14,46,None,None,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 71}"
6,8,145,742,10,47,None,None,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 71}"
7,5,157,730,13,48,None,None,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 71}"
8,5,206,681,13,49,None,None,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 71}"
9,10,147,740,8,50,None,None,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 71}"


###### model performance check by over sampling

In [116]:
params = [
    # [41, None, None],
    # [42, None, None],
    # [43, None, None],
    # [44, None, None],
    # [45, None, None],
    # [46, None, None],
    # [47, None, None],
    # [48, None, None],
    # [49, None, None],
    # [50, None, None],

    # [46, "ROS", 1.0],
    # [46, "ROS", 2.0],
    # [46, "SMOTE", 1.0],
    # [46, "SMOTE", 2.0],
    # [46, "ADASYN", 1.0],
    # [46, "ADASYN", 2.0]

    [46, "ADASYN", 3.0],
    [46, "ADASYN", 4.0],
    [46, "ADASYN", 5.0]
]

In [117]:
split_parameters = {}
for i, param in enumerate(params):
    split_parameter = copy.deepcopy(split_parameter_default)
    split_parameter["random_state"] = param[0]
    split_parameter["sampling_method"] = param[1]
    split_parameter["sampling_ratio"] = param[2]
    split_parameters[i] = split_parameter

# print(split_parameters)

In [118]:
save_path = "data/result/create_train_test/oversample_traindata"
usr_pl_smpl_test_result_ftpn_df_over_sample = run_summary_pl_result_create_train_and_test(split_parameters, save_path)



##############################################################################################################################
# 3) Create Train/Test Split (훈련/테스트 데이터 분할) 
##############################################################################################################################

     훈련 및 테스트 데이터셋 생성 중...
     - 분할 전 필터링 미적용.
     - Feature Generation 미적용.

     - 분할 전 훈련 데이터 클래스 분포: {0.0: 3546, 1.0: 71}
     - ADASYN 오버샘플링 적용 (sampling_ratio: 3.0)
     - 샘플링 적용 후 훈련 데이터 클래스 분포: {0.0: 3546, 1.0: 10611}
      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
    Best F2 (class=1) score (CV): 0.9759

      Forecasting the test dataset...
      Forecasting done!
Best threshold for F2 score: 0.9394 with F2 score: 0.9966
      Calculation of the ROC curve...
      Calculation done
      

In [119]:
usr_pl_smpl_test_result_ftpn_df_over_sample

,fn,fp,tn,tp,random_state,sampling_method_used,sampling_ratio_used,class_distribution_before_sampling,class_distribution_after_sampling
0,6,132,755,12,46,ADASYN,3.0,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 10611}"
1,6,136,751,12,46,ADASYN,4.0,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 14210}"
2,7,120,767,11,46,ADASYN,5.0,"{0.0: 3546, 1.0: 71}","{0.0: 3546, 1.0: 17751}"
